In [ ]:
import nibabel as nib
import numpy as np
import scipy.io as sio
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import os

In [ ]:
# Load atlas

atlas_file = 'path to study specific atlas'
atlas = nib.load(atlas_file)
atlas_data = atlas.get_fdata()
print(f"Atlas dimensions: {atlas_data.shape}")

In [ ]:
# Check number of ROIs

roi_names = []
with open('path to list of ROIs', 'r') as rois_labels:
    for label in rois_labels:
        roi_names.append(label)

print(f"{int(np.max(atlas_data))} ROIs")
print(f"Length of roi_names equals max value of atlas_data: {len(roi_names) == np.max(atlas_data)}")

In [ ]:
# Load gRAICAR results

gRAICAR_result = sio.loadmat('path to gRAICAR result.mat file (in gRAICAR output folder)') 

In [ ]:
gRAICAR_result.keys()

In [ ]:
# Locate "confidence of subject load" values

conf_subj_load = gRAICAR_result['obj'][0, 0][1][0][0][10]
print(conf_subj_load)

In [ ]:
conf_subj_load.shape

In [ ]:
last_comp = conf_subj_load.shape[1]

for i in range(0, last_comp):
    r13 = conf_subj_load[:8, i]
    SHM = conf_subj_load[8:19, i]
    veh = conf_subj_load[19:, i]
    data = np.concatenate([r13, SHM, veh])
    groups = ['r13'] * len(r13) + ['SHM'] * len(SHM) + ['veh'] * len(veh)

    F_stat, p_value = f_oneway(r13, SHM, veh) # ANOVA to determine aligned components with significant group differences
    
    if p_value < 0.05:
        fig, ax = plt.subplots()
        colors = ['green', 'blue', 'red']
        plot = ax.boxplot([r13, SHM, veh], labels = ['r13', 'SHM', 'veh'], patch_artist = True)
        plt.xlabel('Group')
        plt.ylabel('Confidence')
        plt.title(f"Confidence of Subject Loads for Aligned Component {i+1}")
        
        for patch, color in zip(plot['boxes'], colors):
            patch.set_facecolor(color) # fill boxes with color
        plt.show()

        print(f'ANOVA result: F = {F_stat}, p = {p_value}')
        
        tukey = pairwise_tukeyhsd(endog=data, groups=groups, alpha=0.05) # pairiwse Tukey test to determine which groups are significantly different
        print(tukey)

        # identify most prominent ROIs
        comp_maps_dir = 'path to compMaps folder (in gRAICAR output folder)'
        comp_maps = sorted(os.listdir(comp_maps_dir))
        img = nib.load(os.path.join(comp_maps_dir, comp_maps[i]))
        data = img.get_fdata()
        header = img.header
        affine = img.affine
        flattened_data = data.flatten()
        mean = np.mean(flattened_data)
        std = np.std(flattened_data)
        z_scores = (data - mean) / std # z-score normalize data

        signif_rois = set()

        rois = {}
        for x in range(96):
            for y in range(96):
                for z in range(25):
                    rois[(x, y, z)] = z_scores[x, y, z] # roi coordinates: z_score
        sorted_rois = dict(sorted(rois.items(), key=lambda items: items[1], reverse=True)) # sort rois by z_score in descending order
        for key, value in sorted_rois.items():
            if atlas_data[key[0], key[1], key[2]] != 0: # we want to avoid unlabeled coordinates
                signif_rois.add(int(atlas_data[key[0], key[1], key[2]]))
            if len(signif_rois) == 10: # specifies how many ROIs to include, can be changed as needed
                signif_rois = sorted(signif_rois)
                break
                
        print(f"Top {len(signif_rois)} most prominent ROIs (sorted left to right):\n")
        for roi in signif_rois:
            print(roi_names[roi - 1])